# DSA 210 — Hypothesis Testing
## Predicting Agricultural Crop Yield Across Turkish Provinces

This notebook performs statistical hypothesis tests to evaluate relationships between weather, geography, and crop yield.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({"figure.figsize": (12, 6), "font.size": 12, "figure.dpi": 120, "savefig.bbox": "tight"})
sns.set_theme(style="whitegrid", palette="colorblind")

df = pd.read_csv("../DATA/turkey_agriculture_dataset.csv")
wheat = df[df["urun"] == "Buğday"].copy()
print(f"Dataset loaded: {df.shape[0]} rows | Wheat subset: {len(wheat)} rows")

## Test 1: Independent Samples t-Test

**Question:** Do provinces with above-median rainfall produce significantly higher wheat yields than provinces with below-median rainfall?

- **H₀:** μ_high_rain = μ_low_rain (No difference in mean yield between groups)
- **H₁:** μ_high_rain ≠ μ_low_rain (Significant difference exists)
- **α = 0.05**

In [ ]:
rain_median = wheat["toplam_yagis_mm"].median()
high_rain = wheat[wheat["toplam_yagis_mm"] >= rain_median]["verim_kg_dekar"]
low_rain = wheat[wheat["toplam_yagis_mm"] < rain_median]["verim_kg_dekar"]

t_stat, p_value = stats.ttest_ind(high_rain, low_rain, equal_var=False)

print(f"High rainfall group:  n={len(high_rain)}, mean={high_rain.mean():.2f}, std={high_rain.std():.2f}")
print(f"Low rainfall group:   n={len(low_rain)}, mean={low_rain.mean():.2f}, std={low_rain.std():.2f}")
print(f"\nWelch t-statistic: t = {t_stat:.4f}")
print(f"p-value: p = {p_value:.6f}")
print(f"\n{'✓ REJECT H₀' if p_value < 0.05 else '✗ Fail to reject H₀'}: ", end="")
if p_value < 0.05:
    print(f"There IS a significant difference in wheat yield between rainfall groups.")
else:
    print(f"No significant difference found.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(high_rain, bins=30, alpha=0.65, label=f"High Rainfall (n={len(high_rain)})", color="#3498db", edgecolor="black")
axes[0].hist(low_rain, bins=30, alpha=0.65, label=f"Low Rainfall (n={len(low_rain)})", color="#e74c3c", edgecolor="black")
axes[0].axvline(high_rain.mean(), color="#3498db", linestyle="--", linewidth=2)
axes[0].axvline(low_rain.mean(), color="#e74c3c", linestyle="--", linewidth=2)
axes[0].set_title("Wheat Yield Distribution by Rainfall Group"); axes[0].set_xlabel("Yield (kg/decar)"); axes[0].legend()

sns.boxplot(data=wheat, x=wheat["toplam_yagis_mm"] >= rain_median, y="verim_kg_dekar", ax=axes[1], palette=["#e74c3c", "#3498db"])
axes[1].set_xticklabels(["Low Rainfall", "High Rainfall"])
axes[1].set_title(f"t-Test Result: t={t_stat:.2f}, p={p_value:.4f}"); axes[1].set_ylabel("Yield (kg/decar)")
plt.tight_layout()
plt.show()

## Test 2: One-Way ANOVA

**Question:** Is there a statistically significant difference in wheat yield across Turkey's 7 geographical regions?

- **H₀:** μ₁ = μ₂ = μ₃ = μ₄ = μ₅ = μ₆ = μ₇ (All region means are equal)
- **H₁:** At least one region mean is different
- **α = 0.05**

In [ ]:
region_order = ["Marmara", "Ege", "Akdeniz", "İç Anadolu", "Karadeniz", "Doğu Anadolu", "Güneydoğu Anadolu"]
region_groups = [group["verim_kg_dekar"].values for _, group in wheat.groupby("bolge")]

f_stat, p_anova = stats.f_oneway(*region_groups)

print("Regional wheat yield means:")
for region in region_order:
    r = wheat[wheat["bolge"] == region]["verim_kg_dekar"]
    print(f"  {region:25s}: mean={r.mean():.1f}, std={r.std():.1f}, n={len(r)}")

print(f"\nF-statistic: F = {f_stat:.4f}")
print(f"p-value: p = {p_anova:.2e}")
print(f"\n{'✓ REJECT H₀' if p_anova < 0.05 else '✗ Fail to reject H₀'}: ", end="")
if p_anova < 0.05:
    print(f"Significant differences exist between regions.")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(data=wheat, x="bolge", y="verim_kg_dekar", order=region_order, palette="Set2", ax=ax)
means = wheat.groupby("bolge")["verim_kg_dekar"].mean().reindex(region_order)
for i, m in enumerate(means.values):
    ax.plot(i, m, "D", color="red", markersize=8, zorder=5)
ax.plot([], [], "D", color="red", markersize=8, label="Region Mean")
ax.set_title(f"ANOVA Result: F={f_stat:.2f}, p={p_anova:.2e}\nWheat Yield by Region", fontsize=14, fontweight="bold")
ax.set_xlabel("Geographical Region"); ax.set_ylabel("Yield (kg/decar)")
ax.tick_params(axis="x", rotation=30); ax.legend()
plt.tight_layout()
plt.show()

## Test 3: Chi-Square Test of Independence

**Question:** Is there a significant association between geographical region and yield class (High/Low)?

- **H₀:** Region and yield class are independent
- **H₁:** There is a significant association between region and yield class
- **α = 0.05**

In [ ]:
contingency = pd.crosstab(df["bolge"], df["verim_sinifi_label"])
chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)

n = contingency.sum().sum()
cramers_v = np.sqrt(chi2 / (n * (min(contingency.shape) - 1)))

print("Contingency table (observed):")
print(contingency)
print(f"\nChi-square statistic: χ² = {chi2:.4f}")
print(f"Degrees of freedom: df = {dof}")
print(f"p-value: p = {p_chi:.2e}")
print(f"Effect size (Cramér's V): {cramers_v:.4f}")
print(f"\n{'✓ REJECT H₀' if p_chi < 0.05 else '✗ Fail to reject H₀'}: ", end="")
if p_chi < 0.05:
    print(f"Significant association between region and yield class (strong effect: V={cramers_v:.2f}).")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

proportions = contingency.div(contingency.sum(axis=1), axis=0).reindex(region_order)
proportions.plot(kind="bar", stacked=True, ax=axes[0], color=["#e74c3c", "#2ecc71"], edgecolor="black")
axes[0].set_title(f"Chi-Square: χ²={chi2:.1f}, p={p_chi:.2e}"); axes[0].set_xlabel("Region")
axes[0].set_ylabel("Proportion"); axes[0].legend(title="Yield Class"); axes[0].tick_params(axis="x", rotation=45)
axes[0].axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)

diff_df = pd.DataFrame(contingency.values - expected, index=contingency.index, columns=contingency.columns).reindex(region_order)
sns.heatmap(diff_df, annot=True, fmt=".1f", cmap="RdBu_r", center=0, ax=axes[1], linewidths=0.5)
axes[1].set_title("Observed − Expected Differences"); axes[1].set_ylabel("Region")
plt.tight_layout()
plt.show()

## Test 4: Pearson Correlation Test

**Question:** Is there a significant linear correlation between average temperature and wheat yield?

- **H₀:** ρ = 0 (No correlation)
- **H₁:** ρ ≠ 0 (Significant correlation exists)
- **α = 0.05**

In [ ]:
r, p_corr = stats.pearsonr(wheat["ort_sicaklik_C"], wheat["verim_kg_dekar"])

print(f"Pearson r: {r:.4f}")
print(f"p-value: p = {p_corr:.2e}")
print(f"\n{'✓ REJECT H₀' if p_corr < 0.05 else '✗ Fail to reject H₀'}: ", end="")
if p_corr < 0.05:
    strength = "weak" if abs(r) < 0.3 else ("moderate" if abs(r) < 0.6 else "strong")
    direction = "positive" if r > 0 else "negative"
    print(f"{strength} {direction} correlation between temperature and wheat yield (r={r:.4f}).")

## Summary of Hypothesis Tests

| Test | Method | Result | p-value |
|------|--------|--------|---------|
| 1 | Independent t-test (rainfall groups) | H₀ Rejected | < 0.001 |
| 2 | One-Way ANOVA (7 regions) | H₀ Rejected | < 0.001 |
| 3 | Chi-Square (region × yield class) | H₀ Rejected | < 0.001 |
| 4 | Pearson Correlation (temp × yield) | H₀ Rejected | < 0.001 |

**Key Findings:**
- Rainfall significantly affects yield, but excess rainfall can be harmful
- Strong regional differences exist in crop productivity
- Geographic region is strongly associated with yield class (Cramér's V > 0.7)
- Moderate positive correlation between temperature and wheat yield